In [ ]:
from PIL import Image
from tqdm import tqdm
from src.dino.dinov2 import DinoV2
from pathlib import Path
import numpy as np
import csv
from sklearn.manifold import TSNE
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import umap

In [ ]:
CROPS_DIR = Path("/home/pasquale/PycharmProjects/Glomeruli-FP03-2026/data/glomeruli")
OUTPUT_EMBEDS = CROPS_DIR.parent / "dinov2_cls"
OUTPUT_EMBEDS.mkdir(parents=True, exist_ok=True)

IMAGE_SIZE = 518

model = DinoV2(model_name="large", input_size=IMAGE_SIZE)

In [ ]:
image_paths = sorted(path for path in CROPS_DIR.iterdir())

cls_path = OUTPUT_EMBEDS / "cls_embeds.npy"
csv_path = OUTPUT_EMBEDS / "cls_embeds.csv"

embedding_dim = model.backbone.hidden_dim

cls_memmap = np.lib.format.open_memmap(
    cls_path,
    mode="w+",
    dtype="float32",
    shape=(len(image_paths), embedding_dim)
)

with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["index", "image_path"])

    for i, image_path in enumerate(tqdm(image_paths, desc="Extracting CLS embeddings")):
        image = Image.open(image_path).convert("RGB")

        embed = model(image, "cls")
        embed = np.asarray(embed).astype("float32")
        embed = np.squeeze(embed)

        cls_memmap[i] = embed

        writer.writerow([i, str(image_path)])


cls_memmap.flush()

In [ ]:
image_paths = sorted(path for path in CROPS_DIR.iterdir())

patch_mean_path = OUTPUT_EMBEDS / "patch_mean_embeds.npy"
csv_path = OUTPUT_EMBEDS / "patch_mean_embeds.csv"

embedding_dim = model.backbone.hidden_dim

patch_mean_memmap = np.lib.format.open_memmap(
    patch_mean_path,
    mode="w+",
    dtype="float32",
    shape=(len(image_paths), embedding_dim)
)

with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["index", "image_path"])

    for i, image_path in enumerate(tqdm(image_paths, desc="Extracting mean PATCH embeddings")):
        image = Image.open(image_path).convert("RGB")

        patch_embed = model(image, "patch")
        patch_embed = np.asarray(patch_embed).astype("float32")
        patch_embed = np.squeeze(patch_embed)

        # patch_embed ha forma:
        # (n_patches, embedding_dim)
        #
        # Facciamo la media sui patch:
        patch_mean_embed = patch_embed.mean(axis=0)

        # patch_mean_embed ha forma:
        # (embedding_dim,)

        if patch_mean_embed.shape != (embedding_dim,):
            raise ValueError(
                f"Shape inattesa per {image_path}: "
                f"ottenuto {patch_mean_embed.shape}, atteso {(embedding_dim,)}"
            )

        patch_mean_memmap[i] = patch_mean_embed

        writer.writerow([i, str(image_path)])

patch_mean_memmap.flush()

In [ ]:
image_paths = sorted(path for path in CROPS_DIR.iterdir())

patch_stats_path = OUTPUT_EMBEDS / "patch_stats_embeds.npy"
csv_path = OUTPUT_EMBEDS / "patch_stats_embeds.csv"

embedding_dim = model.backbone.hidden_dim
stats_dim = embedding_dim * 2

patch_stats_memmap = np.lib.format.open_memmap(
    patch_stats_path,
    mode="w+",
    dtype="float32",
    shape=(len(image_paths), stats_dim)
)

with open(csv_path, mode="w", newline="", encoding="utf-8") as f:
    writer = csv.writer(f)
    writer.writerow(["index", "image_path"])

    for i, image_path in enumerate(tqdm(image_paths, desc="Extracting PATCH mean+std embeddings")):
        image = Image.open(image_path).convert("RGB")

        patch_embed = model(image, "patch")
        patch_embed = np.asarray(patch_embed).astype("float32")
        patch_embed = np.squeeze(patch_embed)

        # patch_embed ha forma:
        # (n_patches, embedding_dim)

        patch_mean = patch_embed.mean(axis=0)
        patch_std = patch_embed.std(axis=0)

        patch_stats_embed = np.concatenate([
            patch_mean,
            patch_std
        ]).astype("float32")

        # patch_stats_embed ha forma:
        # (embedding_dim * 2,)

        if patch_stats_embed.shape != (stats_dim,):
            raise ValueError(
                f"Shape inattesa per {image_path}: "
                f"ottenuto {patch_stats_embed.shape}, atteso {(stats_dim,)}"
            )

        patch_stats_memmap[i] = patch_stats_embed

        writer.writerow([i, str(image_path)])

patch_stats_memmap.flush()

In [ ]:
X = np.load("/home/pasquale/PycharmProjects/Glomeruli-FP03-2026/data/dinov2_cls/patch_mean_embeds.npy")

In [ ]:
# =========================
# PCA 2D SENZA SCALER
# =========================

pca_raw = PCA(n_components=2)
X_pca_raw = pca_raw.fit_transform(X)

print("Shape PCA 2D senza scaler:", X_pca_raw.shape)
print("Varianza spiegata senza scaler:", pca_raw.explained_variance_ratio_)
print("Varianza cumulata senza scaler:", pca_raw.explained_variance_ratio_.sum())


plt.figure(figsize=(6, 4))

plt.scatter(
    X_pca_raw[:, 0],
    X_pca_raw[:, 1],
    s=8,
    alpha=0.7
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("CLS token DINOv2 - PCA 2D senza StandardScaler")
plt.grid(True, alpha=0.3)

plt.show()


# =========================
# PCA 2D CON SCALER
# =========================

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca_scaled = PCA(n_components=2)
X_pca_scaled = pca_scaled.fit_transform(X_scaled)

print("Shape PCA 2D con scaler:", X_pca_scaled.shape)
print("Varianza spiegata con scaler:", pca_scaled.explained_variance_ratio_)
print("Varianza cumulata con scaler:", pca_scaled.explained_variance_ratio_.sum())


plt.figure(figsize=(6, 4))

plt.scatter(
    X_pca_scaled[:, 0],
    X_pca_scaled[:, 1],
    s=8,
    alpha=0.7
)

plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("CLS token DINOv2 - PCA 2D con StandardScaler")
plt.grid(True, alpha=0.3)

plt.show()

In [ ]:
# ============================================================
# 1) CARICAMENTO EMBEDDING CLS RAW
# ============================================================



print("Shape originale:", X.shape)

# ============================================================
# 2) PCA RAW AL 98%
# ============================================================

pca_98 = PCA(n_components=0.98)
X_pca_98 = pca_98.fit_transform(X)

print("Shape dopo PCA raw 98%:", X_pca_98.shape)
print("Varianza spiegata:", pca_98.explained_variance_ratio_.sum())

# ============================================================
# 3) t-SNE SUI DATI PCA 98%
# ============================================================

tsne = TSNE(
    n_components=2,
    random_state=0,
    perplexity=30,
    init="pca",
    learning_rate="auto"
)

X_tsne = tsne.fit_transform(X_pca_98)

print("Shape t-SNE:", X_tsne.shape)

# ============================================================
# 4) PLOT SENZA LABEL
# ============================================================

plt.figure(figsize=(8, 8))

plt.scatter(
    X_tsne[:, 0],
    X_tsne[:, 1],
    s=5,
    alpha=0.7
)

plt.title("t-SNE su CLS token DINOv2 dopo PCA raw 98%")
plt.xlabel("t-SNE 1")
plt.ylabel("t-SNE 2")
plt.grid(True)
plt.show()

In [ ]:
PCA_VARIANCES = [0.95, 0.98]

UMAP_COMPONENTS = 2
UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_METRIC = "euclidean"
RANDOM_STATE = 42


# ============================================================
# 1) EMBEDDING RAW
# ============================================================

print("Shape originale:", X.shape)


# ============================================================
# 2) PCA RAW + UMAP PER 95% E 98%
# ============================================================

results = {}

for pca_variance in PCA_VARIANCES:

    print("=" * 70)
    print(f"PCA raw {int(pca_variance * 100)}%")

    # ----------------------------
    # PCA RAW
    # ----------------------------
    pca = PCA(
        n_components=pca_variance,
        random_state=RANDOM_STATE
    )

    X_pca = pca.fit_transform(X)

    print(f"Shape dopo PCA raw {int(pca_variance * 100)}%:", X_pca.shape)
    print("Numero componenti PCA:", X_pca.shape[1])
    print("Varianza spiegata cumulata:", pca.explained_variance_ratio_.sum())

    # ----------------------------
    # UMAP 2D
    # ----------------------------
    reducer = umap.UMAP(
        n_components=UMAP_COMPONENTS,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=RANDOM_STATE
    )

    X_umap = reducer.fit_transform(X_pca)

    print("Shape UMAP:", X_umap.shape)

    results[pca_variance] = {
        "pca": pca,
        "X_pca": X_pca,
        "X_umap": X_umap
    }

    # ----------------------------
    # PLOT
    # ----------------------------
    plt.figure(figsize=(8, 8))

    plt.scatter(
        X_umap[:, 0],
        X_umap[:, 1],
        s=5,
        alpha=0.7
    )

    plt.title(
        f"UMAP DINOv2 dopo PCA raw {int(pca_variance * 100)}%"
    )

    plt.xlabel("UMAP 1")
    plt.ylabel("UMAP 2")
    plt.grid(True, alpha=0.3)
    plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score, davies_bouldin_score, calinski_harabasz_score
from sklearn.manifold import trustworthiness

import umap
import hdbscan


# ============================================================
# 1. PCA RAW 98%
# ============================================================

pca = PCA(n_components=0.98, random_state=42)
X_pca_98 = pca.fit_transform(X)

print("Shape iniziale:", X.shape)
print("Shape PCA 98%:", X_pca_98.shape)
print("Componenti PCA:", X_pca_98.shape[1])
print("Varianza cumulata:", pca.explained_variance_ratio_.sum())


# ============================================================
# 2. TEST DI DIVERSI n_components UMAP
# ============================================================

n_components_list = range(2, X_pca_98.shape[1])

results = []

for n_comp in n_components_list:
    print("=" * 70)
    print(f"UMAP n_components = {n_comp}")

    reducer = umap.UMAP(
        n_components=n_comp,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        random_state=42
    )

    X_umap_k = reducer.fit_transform(X_pca_98)

    clusterer = hdbscan.HDBSCAN(
        min_cluster_size=20,
        min_samples=10,
        metric="euclidean",
        cluster_selection_method="eom"
    )

    labels = clusterer.fit_predict(X_umap_k)

    unique_labels = sorted(set(labels))
    n_clusters = len(unique_labels) - (1 if -1 in labels else 0)
    noise_ratio = np.mean(labels == -1)

    print("Numero cluster:", n_clusters)
    print("Noise ratio:", noise_ratio)
    print("Label:", unique_labels)

    # Considero solo i punti non noise per le metriche interne
    mask = labels != -1

    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X_umap_k[mask], labels[mask])
        db = davies_bouldin_score(X_umap_k[mask], labels[mask])
        ch = calinski_harabasz_score(X_umap_k[mask], labels[mask])
    else:
        sil = np.nan
        db = np.nan
        ch = np.nan

    # Trustworthiness: misura quanto UMAP conserva i vicini locali dello spazio PCA
    trust = trustworthiness(
        X_pca_98,
        X_umap_k,
        n_neighbors=15
    )

    results.append({
        "umap_n_components": n_comp,
        "n_clusters": n_clusters,
        "noise_ratio": noise_ratio,
        "silhouette": sil,
        "davies_bouldin": db,
        "calinski_harabasz": ch,
        "trustworthiness": trust
    })


results_df = pd.DataFrame(results)
print(results_df)

In [ ]:
# X_umap deve essere il risultato:
# patch mean DINOv2 → PCA raw 98% → UMAP 2D

min_cluster_sizes = [8, 10, 12, 15, 20, 25, 30]
min_samples_list = [3, 5, 8, 10, 15]

results = []

for min_cluster_size in min_cluster_sizes:
    for min_samples in min_samples_list:

        clusterer = hdbscan.HDBSCAN(
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric="euclidean",
            cluster_selection_method="eom"
        )

        labels = clusterer.fit_predict(X_umap)

        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        noise_ratio = np.mean(labels == -1)

        mask = labels != -1

        if n_clusters >= 2 and mask.sum() > n_clusters:
            sil = silhouette_score(X_umap[mask], labels[mask])
            db = davies_bouldin_score(X_umap[mask], labels[mask])
            ch = calinski_harabasz_score(X_umap[mask], labels[mask])
        else:
            sil = np.nan
            db = np.nan
            ch = np.nan

        results.append({
            "min_cluster_size": min_cluster_size,
            "min_samples": min_samples,
            "n_clusters": n_clusters,
            "noise_ratio": noise_ratio,
            "silhouette": sil,
            "davies_bouldin": db,
            "calinski_harabasz": ch
        })

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by=["noise_ratio", "silhouette"],
    ascending=[True, False]
)

print(results_df)

good = results_df[
    (results_df["n_clusters"] >= 4) &
    (results_df["n_clusters"] <= 10) &
    (results_df["noise_ratio"] <= 0.15)
].sort_values(
    by=["silhouette", "davies_bouldin"],
    ascending=[False, True]
)

print(good)

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(results_df["umap_n_components"], results_df["n_clusters"], marker="o")
plt.xlabel("UMAP n_components")
plt.ylabel("Numero cluster")
plt.title("Numero di cluster al variare di n_components UMAP")
plt.grid(True, alpha=0.3)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(results_df["umap_n_components"], results_df["noise_ratio"], marker="o")
plt.xlabel("UMAP n_components")
plt.ylabel("Noise ratio")
plt.title("Percentuale di noise al variare di n_components UMAP")
plt.grid(True, alpha=0.3)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(results_df["umap_n_components"], results_df["silhouette"], marker="o")
plt.xlabel("UMAP n_components")
plt.ylabel("Silhouette score")
plt.title("Silhouette al variare di n_components UMAP")
plt.grid(True, alpha=0.3)
plt.show()


plt.figure(figsize=(8, 5))
plt.plot(results_df["umap_n_components"], results_df["trustworthiness"], marker="o")
plt.xlabel("UMAP n_components")
plt.ylabel("Trustworthiness")
plt.title("Conservazione dei vicini locali")
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import umap
import hdbscan

from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)


# ============================================================
# PARAMETRI PRINCIPALI DA MODIFICARE
# ============================================================

PCA_VARIANCE = 0.98          # esempio: 0.95 oppure 0.98
UMAP_COMPONENTS = 7          # esempio: 2, 3, 4, 5, 7

UMAP_N_NEIGHBORS = 15
UMAP_MIN_DIST = 0.1
UMAP_METRIC = "euclidean"
RANDOM_STATE = 42

HDBSCAN_MIN_CLUSTER_SIZE = 20
HDBSCAN_MIN_SAMPLES = 15
HDBSCAN_METRIC = "euclidean"
HDBSCAN_CLUSTER_SELECTION_METHOD = "eom"


# ============================================================
# 1. PCA RAW
# ============================================================

print("Shape iniziale:", X.shape)

pca = PCA(
    n_components=PCA_VARIANCE,
    random_state=RANDOM_STATE
)

X_pca = pca.fit_transform(X)

print(f"Shape PCA {int(PCA_VARIANCE * 100)}%:", X_pca.shape)
print("Componenti PCA:", X_pca.shape[1])
print("Varianza cumulata:", np.sum(pca.explained_variance_ratio_))


# ============================================================
# 2. UMAP
# ============================================================

reducer = umap.UMAP(
    n_components=UMAP_COMPONENTS,
    n_neighbors=UMAP_N_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    metric=UMAP_METRIC,
    random_state=RANDOM_STATE
)

X_umap = reducer.fit_transform(X_pca)

print(f"Shape UMAP {UMAP_COMPONENTS}D:", X_umap.shape)


# ============================================================
# 3. FUNZIONE DI VALUTAZIONE HDBSCAN
# ============================================================

def evaluate_hdbscan(X_cluster, labels, name):
    n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
    noise_ratio = np.mean(labels == -1)

    print("=" * 70)
    print(name)
    print("Numero cluster:", n_clusters)
    print("Noise ratio:", noise_ratio)
    print("Label:", sorted(set(labels)))

    mask = labels != -1

    if n_clusters >= 2 and mask.sum() > n_clusters:
        sil = silhouette_score(X_cluster[mask], labels[mask])
        db = davies_bouldin_score(X_cluster[mask], labels[mask])
        ch = calinski_harabasz_score(X_cluster[mask], labels[mask])

        print("Silhouette:", sil)
        print("Davies-Bouldin:", db)
        print("Calinski-Harabasz:", ch)
    else:
        print("Metriche non calcolabili")


# ============================================================
# 5. HDBSCAN SU UMAP
# ============================================================

clusterer_umap = hdbscan.HDBSCAN(
    min_cluster_size=HDBSCAN_MIN_CLUSTER_SIZE,
    min_samples=HDBSCAN_MIN_SAMPLES,
    metric=HDBSCAN_METRIC,
    cluster_selection_method=HDBSCAN_CLUSTER_SELECTION_METHOD
)

labels_umap = clusterer_umap.fit_predict(X_umap)

evaluate_hdbscan(
    X_cluster=X_umap,
    labels=labels_umap,
    name=f"PCA {int(PCA_VARIANCE * 100)}% → UMAP {UMAP_COMPONENTS}D → HDBSCAN"
)


# ============================================================
# 6. CREAZIONE UMAP 2D SOLO PER VISUALIZZARE
# ============================================================

if UMAP_COMPONENTS == 2:
    X_umap_2d = X_umap
else:
    reducer_2d = umap.UMAP(
        n_components=2,
        n_neighbors=UMAP_N_NEIGHBORS,
        min_dist=UMAP_MIN_DIST,
        metric=UMAP_METRIC,
        random_state=RANDOM_STATE
    )

    X_umap_2d = reducer_2d.fit_transform(X_pca)


# ============================================================
# 8. VISUALIZZAZIONE LABELS UMAP-HDBSCAN SU UMAP 2D
# ============================================================

plt.figure(figsize=(9, 7))

scatter = plt.scatter(
    X_umap_2d[:, 0],
    X_umap_2d[:, 1],
    c=labels_umap,
    s=12,
    alpha=0.85,
    cmap="tab20"
)

plt.xlabel("UMAP 1")
plt.ylabel("UMAP 2")

plt.title(
    f"HDBSCAN su PCA {int(PCA_VARIANCE * 100)}% → "
    f"UMAP {UMAP_COMPONENTS}D"
)

plt.colorbar(scatter, label="Cluster")
plt.grid(True, alpha=0.3)
plt.show()


# ============================================================
# 9. DISTRIBUZIONE CLUSTER
# ============================================================

print("=" * 70)
print("Distribuzione cluster HDBSCAN su UMAP")

unique_labels, counts = np.unique(labels_umap, return_counts=True)

for label, count in zip(unique_labels, counts):
    if label == -1:
        print(f"Noise: {count}")
    else:
        print(f"Cluster {label}: {count}")

In [ ]:
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection="3d")

scatter = ax.scatter(
    X_umap[:, 0],
    X_umap[:, 1],
    X_umap[:, 2],
    c=labels_umap,
    s=12,
    alpha=0.85,
    cmap="tab20"
)

ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
ax.set_zlabel("UMAP 3")

ax.set_title("HDBSCAN su PCA 98% → UMAP 3D")

fig.colorbar(scatter, ax=ax, label="Cluster")
plt.show()

In [ ]:
image_paths = np.array(image_paths)

def plot_cluster_images(labels, cluster_id, max_images=12):
    idx_cluster = np.where(labels == cluster_id)[0]

    print(f"Cluster {cluster_id}: {len(idx_cluster)} immagini")

    selected = idx_cluster[:max_images]

    n_cols = 4
    n_rows = int(np.ceil(len(selected) / n_cols))

    fig, axes = plt.subplots(n_rows, n_cols, figsize=(12, 3 * n_rows))
    axes = np.array(axes).ravel()

    for ax in axes:
        ax.axis("off")

    for ax, idx in zip(axes, selected):
        img = Image.open(image_paths[idx]).convert("RGB")
        ax.imshow(img)
        ax.set_title(f"idx={idx}\ncluster={cluster_id}", fontsize=9)
        ax.axis("off")

    plt.suptitle(f"Cluster {cluster_id} - n={len(idx_cluster)}")
    plt.tight_layout()
    plt.show()


for cluster_id in sorted(set(labels_umap2)):
    if cluster_id == -1:
        continue

    plot_cluster_images(labels_umap2, cluster_id, max_images=12)